# GE 对接 TensorFlow

TensorFlow 模型接入昇腾分为在线和离线两类。在线路径由 TF Adapter 在训练/在线推理时驱动 GE；离线路径又分为 Parser + Build API 和 ATC CLI 两种入口：前者调用 Parser（`aclgrphParseTensorFlow`）得到内存中的 AscendIR `Graph`，再用 GE Build API 编译保存 OM；后者让 ATC CLI 直接重新读取 `.pb` 编译。Parser API 产出的内存 `Graph` 不会被 `atc --model=model.pb` 直接消费。

本节讲清这两类路径、Parser 接口的用法与配置参数，并给出「TorchAir / Parser / 手工构图」三种接入方式的选型矩阵。

本节学习大纲如下：

- 在线和离线两类接入路径总览（TF Adapter / Parser + Build API / ATC CLI）
- TF Adapter 在线路径
- Parser 离线路径：`aclgrphParseTensorFlow`
- 三阶段实践：Parser → Build API → OM → ACL/NPU
- Parser 支持的配置参数
- 接入约束与常见问题
- 接入路径选型矩阵（TorchAir / Parser / 手工构图）

> 说明：接口签名、配置参数取自 GE 文档；具体产品支持范围与版本差异以你所用版本文档为准。

## 1. 在线和离线两类接入路径总览

<p align="left"><img src="./images/tensorflow_paths.svg" alt="TensorFlow 接入 GE 的两类路径" width="110%"></p>

| 路径 | 适配组件 | 输入 | 编译时机 | 典型场景 |
| --- | --- | --- | --- | --- |
| 在线：TF Adapter | TF Adapter | TF 脚本（GraphDef / tf.function） | 运行时由框架驱动 GE 在线编译执行 | 训练、在线推理（框架内运行） |
| 离线：Parser + Build API | `aclgrphParseTensorFlow` + `aclgrphBuildModel` / `aclgrphSaveModel` | 已导出的 `.pb` 模型 | 代码内先解析为 `Graph`，再由 Build API 编译并保存 OM | 需要在代码中检查或修改 Graph 的部署 |
| 离线：ATC CLI | `atc`（TensorFlow 使用 `--framework=3`） | 已导出的 `.pb` 模型 | ATC 直接重新读取 `.pb` 并编译为 OM | 固定结构模型的快速离线部署 |

> 回顾第一章「框架驱动路径」：对 TensorFlow，`TF Adapter` 负责把 TF GraphDef（TF 1.15）或 `tf.function` 修饰的函数（TF 2.6.5）转换为 AscendIR，再交给 GE 编译执行。Parser + Build API 与 ATC CLI 都是离线入口，但二者是并列替代关系，不是 Parser 之后再把同一个 `Graph` 交给 ATC。

## 2. TF Adapter 在线路径

TF Adapter 是 TensorFlow 框架的昇腾适配组件，在 TF 运行时把计算图透明转换为 AscendIR 并交由 GE 编译执行：

| TF 版本 | 转换对象 | 说明 |
| --- | --- | --- |
| TensorFlow 1.15 | TF GraphDef | 经典 Graph + Session 模式，整图交给 GE |
| TensorFlow 2.6.5 | `tf.function` 修饰的函数 | Eager 默认，需用 `@tf.function` 把函数构造成图 |

在线路径的特点：

- **框架运行时驱动**：用户像平常一样写 TF 脚本，由 Adapter + GE 在后端完成图编译执行，无需手工调用 ATC；
- **适合训练**：与 GeSession 在线路径一致，可支持迭代执行；
- **依赖框架运行时**：执行时需要 TF + 昇腾适配运行时在位。

> 在线路径下，用户通常不直接接触 AscendIR；GE 作为后端透明运行。本节后续重点放在**用户可显式调用**的离线 Parser 路径。

## 3. 离线路径：aclgrphParseTensorFlow

当你已经导出了 TensorFlow 的 `.pb` 模型，想走「离线编译 + 独立部署」时，可以选择下面两条相互独立的路线：

1. **Parser + Build API**：`aclgrphParseTensorFlow` 把 `.pb` 解析为进程内的 `ge::Graph`，再依次调用 `aclgrphBuildInitialize`、`aclgrphBuildModel`，需要落盘时再调用 `aclgrphSaveModel`（最后调用 `aclgrphBuildFinalize`）。
2. **ATC CLI**：直接执行 `atc --model=model.pb --framework=3 --output=model --soc_version=<soc>`；ATC 会重新读取并解析 `.pb`，不会读取另一个进程中 Parser 生成的 `ge::Graph`。

### 接口信息

| 项 | 内容 |
| --- | --- |
| 头文件 | Parser：`#include <parser/tensorflow_parser.h>`；Build：`#include <ge/ge_ir_build.h>` |
| 库文件 | Parser：`libfmk_parser.so`；Build：`libge_compiler.so` |
| 功能 | 将 TensorFlow 模型解析为 `Graph` |

### 函数原型

```cpp
// 基础版：仅传入模型文件路径，输出 Graph
graphStatus aclgrphParseTensorFlow(const char *model_file, ge::Graph &graph);

// 带配置版：附加 parser_params 配置 map
graphStatus aclgrphParseTensorFlow(const char *model_file,
                                   const std::map<ge::AscendString, ge::AscendString> &parser_params,
                                   ge::Graph &graph);
```

### 参数与返回值

| 参数 | 输入/输出 | 说明 |
| --- | --- | --- |
| `model_file` | 输入 | TensorFlow 原始模型文件路径（`.pb`） |
| `parser_params` | 输入 | 配置参数 map，key 为参数类型、value 为参数值，均为 `AscendString` 格式（见第 4 节） |
| `graph` | 输出（内存） | 解析后生成的 AscendIR `Graph`；该对象不会自动写入文件，也不会被 ATC CLI 直接接收 |

返回值类型为 `graphStatus`：`GRAPH_SUCCESS(0)` 表示成功，其他值表示失败。

> **约束**：使用该接口解析得到的 Graph，其 **Graph 名中会包含时间戳**，因此多次调用该接口，Graph 名会不同。如需稳定的图名，可通过 `OUTPUT` 配置项指定转图后的计算图名称。

## 4. Parser 支持的配置参数

`parser_params` 通过 `ge::ir_option::` 下的键设置（key/value 均为 `AscendString`）。常用参数如下：

| 参数 | 作用 | 备注 |
| --- | --- | --- |
| `INPUT_SHAPE` | 指定模型输入 shape；支持静态 shape、shape 范围（动态）、标量 | 不设置则读 Data 节点自带 shape；设置则以此为准并刷新 |
| `INPUT_DATA_NAMES` | 指定输入节点 name 与 index 的映射顺序 | 按输入 name 顺序设置 index |
| `INPUT_FP16_NODES` | 指定输入数据类型为 FP16 的输入节点名 | 多个节点用英文分号分隔 |
| `IS_INPUT_ADJUST_HW_LAYOUT` | 输入是否为 FP16 + NC1HWC0 | 需与 `INPUT_FP16_NODES` 配合 |
| `OUT_NODES` | 指定某层算子作为输出 / 指定输出名称 | 不指定则默认取最后一层 |
| `IS_OUTPUT_ADJUST_HW_LAYOUT` | 输出是否为 FP16 + NC1HWC0 | 需与 `OUT_NODES` 配合 |
| `OUTPUT` | 指定转图后计算图名称 | 可用于规避「图名含时间戳」 |
| `ENABLE_SCOPE_FUSION_PASSES` | 指定生效的 scope 融合规则列表 | **仅 `aclgrphParseTensorFlow` 支持** |

### 使用示意（带配置）

```cpp
#include <parser/tensorflow_parser.h>
#include "ge/ge_api.h"

std::map<ge::AscendString, ge::AscendString> parser_params = {
    // 多输入静态 shape：input_name:n,c,h,w，分号分隔
    {ge::AscendString(ge::ir_option::INPUT_SHAPE),
     ge::AscendString("input_0_0:16,32,208,208;input_1_0:16,64,208,208")},
    // 指定输出节点
    {ge::AscendString(ge::ir_option::OUT_NODES), ge::AscendString("add_input:0")},
    // 指定转图后计算图名称（规避图名含时间戳）
    {ge::AscendString(ge::ir_option::OUTPUT), ge::AscendString("my_tf_graph")},
};

ge::Graph graph;
auto ret = aclgrphParseTensorFlow("model.pb", parser_params, graph);
if (ret != GRAPH_SUCCESS) {
    // 解析失败，结合 GE 日志/错误码定位（见 5.4 节）
}
```

### INPUT_SHAPE 的三种形态

| 形态 | 写法示例 | 含义 |
| --- | --- | --- |
| 静态 shape | `"input_0_0:16,32,208,208"` | 形状完全固定 |
| shape 范围（动态） | `"input_0_0:1~10,32,208,208"` 或某维 `-1` | 维度可变，`-1` 表示 ≥0 任意取值（受内存限制） |
| 标量 | `"input_name1:;input_name2:16,32,208,208"` | 标量输入配置为空 |

> 解析完成后得到的 `Graph` 与第一章「构图三种方式」产出的是同一种 AscendIR 图。若继续走代码内编译，应接 `aclgrphBuildModel` / `aclgrphSaveModel`；若改用 ATC，则 ATC 直接读取原始 `.pb`，二者不要串写成一条调用链。

### 4.1 阶段一：GraphDef → Parser → AscendIR Graph（主机侧）

这一步只验证 TensorFlow Parser：把一个二进制 `GraphDef`（`.pb`）解析成进程内的 AscendIR `Graph`，并输出 readable 图描述。它需要真实的 CANN 主机库和 OPP，但不调用 ATC、Build API 或 ACL，因此不需要物理 NPU。

本单元会把后续阶段需要的 `model_path`、Parser 配置、CANN 库路径保留在 Notebook Python 环境中，请按顺序执行三个阶段。

运行前先 `source` 同一套 CANN 9.0.0 的 `set_env.sh`；阶段二优先使用 `TF_SOC_VERSION` / `SOC_VERSION`，未设置且存在 NPU 时会通过 ACL 自动查询目标 SoC；阶段三会在 cell 内固定启用 NPU 执行，无需手动设置开关。

```bash
source /path/to/cann-9.0.0/set_env.sh
# 有 NPU 时无需设置，阶段二默认查询 0 号设备；可用 TF_DEVICE_ID 改设备号
# 无卡编译或需要覆盖自动检测结果时：export TF_SOC_VERSION=Ascend910B1
# 阶段三会在 cell 内设置 TF_RUN_NPU=1，无需手动 export
# 使用自己的 .pb 时还需设置 TF_MODEL_PATH、TF_INPUT_DATA_NAMES、TF_INPUT_SHAPE、TF_OUT_NODES
```

> 若设置了 `TF_MODEL_PATH`，阶段一和阶段二可以解析/编译自有模型；阶段三的数值对拍只针对本单元内置的二输入 Add 图。

In [ ]:
# === 阶段 1：TensorFlow GraphDef -> aclgrphParseTensorFlow -> AscendIR dump（主机侧） ===
import base64
import os
import shutil
import subprocess
import textwrap
from pathlib import Path

if not os.environ.get("ASCEND_HOME_PATH"):
    raise RuntimeError("请先 source CANN 的 set_env.sh，并设置 ASCEND_HOME_PATH。")
if shutil.which("g++") is None:
    raise RuntimeError("本例需要 g++，请在 CANN 开发环境中运行。")

ascend_home = Path(os.environ["ASCEND_HOME_PATH"]).expanduser()
prefix_candidates = [ascend_home, ascend_home / "x86_64-linux"]
prefix = next(
    (
        candidate
        for candidate in prefix_candidates
        if (candidate / "include/parser/tensorflow_parser.h").is_file()
        and (candidate / "lib64/libfmk_parser.so").is_file()
    ),
    None,
)
if prefix is None:
    raise RuntimeError(
        "在 ASCEND_HOME_PATH 下找不到 parser/tensorflow_parser.h 或 libfmk_parser.so。"
    )

opp_candidates = []
configured_opp = os.environ.get("ASCEND_OPP_PATH")
if configured_opp:
    opp_candidates.append(Path(configured_opp).expanduser())
opp_candidates.extend([ascend_home / "opp", prefix / "opp"])
opp_path = next((candidate for candidate in opp_candidates if candidate.is_dir()), None)
if opp_path is None:
    raise RuntimeError("找不到 OPP 目录，请确认 CANN toolkit + ops 已安装。")

workdir = Path("/tmp/ge_tf_parser_demo")
workdir.mkdir(parents=True, exist_ok=True)
cpp_source = textwrap.dedent(
    r'''
    #include <fstream>
    #include <iostream>
    #include <map>
    #include <sstream>
    #include <string>

    #include "ge/ge_api.h"
    #include "graph/graph.h"
    #include "parser/tensorflow_parser.h"

    int main(int argc, char **argv) {
      if (argc != 7) {
        std::cerr << "usage: parser_demo MODEL INPUT_SHAPE OUT_NODES GRAPH_NAME DUMP_FILE INPUT_DATA_NAMES\n";
        return 2;
      }
      std::map<ge::AscendString, ge::AscendString> parser_params = {
          {ge::AscendString(ge::ir_option::INPUT_DATA_NAMES), ge::AscendString(argv[6])},
          {ge::AscendString(ge::ir_option::INPUT_SHAPE), ge::AscendString(argv[2])},
          {ge::AscendString(ge::ir_option::OUT_NODES), ge::AscendString(argv[3])},
          {ge::AscendString(ge::ir_option::OUTPUT), ge::AscendString(argv[4])},
      };
      ge::Graph graph;
      const auto status = ge::aclgrphParseTensorFlow(argv[1], parser_params, graph);
      ge::AscendString graph_name;
      (void)graph.GetName(graph_name);
      std::cout << "graph_status=" << status << "\n";
      std::cout << "graph_name=" << graph_name.GetString() << "\n";
      if (status != ge::GRAPH_SUCCESS) {
        return 1;
      }
      std::ostringstream readable;
      const auto dump_status = graph.Dump(ge::Graph::DumpFormat::kReadable, readable);
      if (dump_status != ge::GRAPH_SUCCESS) {
        std::cerr << "Graph::Dump failed, status=" << dump_status << "\n";
        return 1;
      }
      std::ofstream output(argv[5]);
      if (!output) {
        std::cerr << "cannot open dump file: " << argv[5] << "\n";
        return 1;
      }
      output << readable.str();
      std::cout << readable.str();
      return 0;
    }
    '''
)
source_path = workdir / "parser_demo.cpp"
binary_path = workdir / "parser_demo"
source_path.write_text(cpp_source, encoding="utf-8")
libdir = prefix / "lib64"
compile_cmd = [
    "g++",
    "-std=c++17",
    "-O2",
    "-D_GLIBCXX_USE_CXX11_ABI=0",
    "-Dgoogle=ascend_private",
    "-I" + str(prefix / "include"),
    str(source_path),
    "-L" + str(libdir),
    "-Wl,-rpath," + str(libdir),
    "-lfmk_parser",
    "-lgraph",
    "-lgraph_base",
    "-lge_common_base",
    "-o",
    str(binary_path),
]
compile_result = subprocess.run(compile_cmd, text=True, capture_output=True)
if compile_result.returncode != 0:
    print(compile_result.stdout)
    print(compile_result.stderr)
    raise RuntimeError("Parser 示例编译失败。")

is_builtin_tf_add = not bool(os.environ.get("TF_MODEL_PATH"))
model_path = os.environ.get("TF_MODEL_PATH")
if model_path:
    model_path = Path(model_path).expanduser().resolve()
    if not model_path.is_file():
        raise FileNotFoundError("TF_MODEL_PATH 不存在：{}".format(model_path))
else:
    model_path = workdir / "tf_add.pb"
    # 内置 GraphDef：两个 DT_FLOAT Placeholder 与一个 DT_FLOAT Add。
    # 原始 shape 为 [1]，下面通过 TF_INPUT_SHAPE 覆盖为 [2, 3]。
    model_path.write_bytes(
        base64.b64decode(
            "CjgKC1BsYWNlaG9sZGVyEgtQbGFjZWhvbGRlcioLCgVkdHlwZRICMAEqDwoFc2hhcGUSBjoEEgIIAQo6Cg1QbGFjZWhvbGRlcl8xEgtQbGFjZWhvbGRlcioLCgVkdHlwZRICMAEqDwoFc2hhcGUSBjoEEgIIAQo2CgphZGRfdGVzdF8xEgNBZGQaC1BsYWNlaG9sZGVyGg1QbGFjZWhvbGRlcl8xKgcKAVQSAjABIgMIhgE="
        )
    )

input_shape = os.environ.get("TF_INPUT_SHAPE", "Placeholder:2,3;Placeholder_1:2,3")
out_nodes = os.environ.get("TF_OUT_NODES", "add_test_1:0")
input_data_names = os.environ.get("TF_INPUT_DATA_NAMES", "Placeholder,Placeholder_1")
graph_name = os.environ.get("TF_GRAPH_NAME", "tf_add_demo")
dump_path = workdir / "tf_add_demo.readable.txt"
run_env = os.environ.copy()
run_env.setdefault("ASCEND_OPP_PATH", str(opp_path))
run_env["LD_LIBRARY_PATH"] = str(libdir) + os.pathsep + run_env.get("LD_LIBRARY_PATH", "")
run_result = subprocess.run(
    [
        str(binary_path),
        str(model_path),
        input_shape,
        out_nodes,
        graph_name,
        str(dump_path),
        input_data_names,
    ],
    text=True,
    capture_output=True,
    env=run_env,
)
if run_result.returncode != 0:
    print(run_result.stdout)
    print(run_result.stderr)
    raise RuntimeError("aclgrphParseTensorFlow 运行失败。")
dump_text = dump_path.read_text(encoding="utf-8")
assert "graph_status=0" in run_result.stdout
if is_builtin_tf_add:
    assert 'graph("{}")'.format(graph_name) in dump_text
    assert "add_test_1" in dump_text
print(run_result.stdout)
print("dump 文件：", dump_path)
print("[OK] TensorFlow GraphDef 已解析为 AscendIR Graph")

### 4.2 阶段二：Parser + Build API → OM（主机侧离线编译）

Parser 生成的 `ge::Graph` 只存在于调用进程内，不能跨进程直接传给另一个编译程序。因此本阶段在同一个 C++ 驱动中重新解析 `.pb`，然后立刻调用：

~~~text
aclgrphParseTensorFlow
    → aclgrphBuildInitialize
    → aclgrphBuildModel
    → aclgrphSaveModel
~~~

这一步生成目标芯片对应的 `tf_add.om`。目标 SoC 按以下优先级确定：`TF_SOC_VERSION` → `SOC_VERSION` → ACL 查询当前 NPU（默认设备 0，可用 `TF_DEVICE_ID` 指定）。自动查询使用 CANN Python ACL 的 `acl.get_soc_name()`；若当前环境没有物理 NPU，则无法推断未来执行机器的型号，仍需显式设置 `TF_SOC_VERSION`。该值应与阶段三实际执行设备匹配。同时需要完整的 CANN toolkit/compiler、TBE Python 组件和 OPP；只有 Parser 运行库的精简环境不能完成 Build API 初始化。

In [ ]:
# === 阶段 2：Parser + GE Build API -> OM（环境变量优先，有卡时自动查询目标 SOC） ===
if "prefix" not in globals() or "model_path" not in globals():
    raise RuntimeError("请先按顺序执行阶段一，准备 Parser 环境和 GraphDef 文件。")


def _normalize_soc_version(value):
    if isinstance(value, (tuple, list)):
        text_values = [item for item in value if isinstance(item, (str, bytes))]
        value = text_values[-1] if text_values else None
    if value is None:
        return None
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    value = str(value).strip()
    if not value:
        return None
    if value.lower().startswith("ascend"):
        return "Ascend" + value[len("Ascend") :]
    return "Ascend" + value


def _detect_soc_version_from_acl(device_id):
    try:
        import acl
    except (ImportError, OSError) as error:
        return None, "无法导入 Python ACL：{}".format(error)

    acl_success = 0
    acl_repeat_initialize = getattr(acl, "ACL_ERROR_REPEAT_INITIALIZE", 100002)
    initialized_here = False
    selected_here = False
    try:
        init_ret = acl.init()
        if init_ret not in (acl_success, acl_repeat_initialize):
            return None, "acl.init 返回 {}".format(init_ret)
        initialized_here = init_ret == acl_success

        # 本 cell 自己初始化 ACL 时，显式选择目标设备；若 Kernel 中 ACL
        # 已由其他代码初始化，则不改动它现有的设备/生命周期。
        if initialized_here:
            set_ret = acl.rt.set_device(device_id)
            if set_ret != acl_success:
                return None, "acl.rt.set_device({}) 返回 {}".format(device_id, set_ret)
            selected_here = True

        soc_name = _normalize_soc_version(acl.get_soc_name())
        if not soc_name:
            return None, "acl.get_soc_name() 未返回有效芯片型号"
        return soc_name, None
    except Exception as error:
        return None, "ACL 自动查询异常：{}".format(error)
    finally:
        if selected_here:
            try:
                reset_ret = acl.rt.reset_device(device_id)
                if reset_ret != acl_success:
                    print("[WARN] acl.rt.reset_device({}) 返回 {}".format(device_id, reset_ret))
            except Exception as cleanup_error:
                print("[WARN] ACL reset_device 清理异常：", cleanup_error)
        if initialized_here:
            try:
                finalize_ret = acl.finalize()
                if finalize_ret != acl_success:
                    print("[WARN] acl.finalize 返回", finalize_ret)
            except Exception as cleanup_error:
                print("[WARN] ACL finalize 清理异常：", cleanup_error)


configured_soc = os.environ.get("TF_SOC_VERSION") or os.environ.get("SOC_VERSION")
if configured_soc:
    soc_version = _normalize_soc_version(configured_soc)
    if not soc_version:
        raise RuntimeError("TF_SOC_VERSION / SOC_VERSION 不是有效的芯片型号。")
    soc_source = (
        "环境变量 TF_SOC_VERSION"
        if os.environ.get("TF_SOC_VERSION")
        else "环境变量 SOC_VERSION"
    )
else:
    try:
        tf_device_id = int(os.environ.get("TF_DEVICE_ID", "0"))
        if tf_device_id < 0:
            raise ValueError
    except ValueError:
        raise RuntimeError("TF_DEVICE_ID 必须是非负整数。")

    soc_version, detect_error = _detect_soc_version_from_acl(tf_device_id)
    if not soc_version:
        raise RuntimeError(
            "未设置 TF_SOC_VERSION / SOC_VERSION，且无法从当前 NPU 自动查询目标芯片"
            "（{}）。有卡环境请确认 Python ACL 与设备可用；无卡编译请显式设置 "
            "TF_SOC_VERSION，例如 Ascend910B1。".format(detect_error)
        )
    soc_source = "ACL 自动查询设备 {}".format(tf_device_id)

# 记录到当前 Notebook Kernel，便于重跑阶段二和后续 cell 查看。
os.environ["TF_SOC_VERSION"] = soc_version
print("阶段二目标 SoC：{}（来源：{}）".format(soc_version, soc_source))

opp_path = globals().get("opp_path", Path(os.environ["ASCEND_HOME_PATH"]) / "opp")
if not opp_path.is_dir():
    raise RuntimeError("找不到 OPP 目录，请检查 CANN toolkit + ops 安装。")
library_dirs = [prefix / "lib64"]
root_libdir = Path(os.environ["ASCEND_HOME_PATH"]) / "lib64"
if root_libdir.is_dir() and root_libdir not in library_dirs:
    library_dirs.append(root_libdir)
rpath = os.pathsep.join(str(path) for path in library_dirs)

build_source = textwrap.dedent(
    r'''
    #include <iostream>
    #include <map>
    #include <string>
    #include "ge/ge_api.h"
    #include "ge/ge_ir_build.h"
    #include "graph/graph.h"
    #include "parser/tensorflow_parser.h"

    int main(int argc, char **argv) {
      if (argc != 8) {
        std::cerr << "usage: tf_build_demo MODEL INPUT_SHAPE OUT_NODES GRAPH_NAME INPUT_DATA_NAMES SOC_VERSION OUTPUT_STEM\n";
        return 2;
      }
      std::map<ge::AscendString, ge::AscendString> parser_params = {
          {ge::AscendString(ge::ir_option::INPUT_DATA_NAMES), ge::AscendString(argv[5])},
          {ge::AscendString(ge::ir_option::INPUT_SHAPE), ge::AscendString(argv[2])},
          {ge::AscendString(ge::ir_option::OUT_NODES), ge::AscendString(argv[3])},
          {ge::AscendString(ge::ir_option::OUTPUT), ge::AscendString(argv[4])},
      };
      ge::Graph graph;
      auto ret = ge::aclgrphParseTensorFlow(argv[1], parser_params, graph);
      if (ret != ge::GRAPH_SUCCESS) {
        std::cerr << "aclgrphParseTensorFlow failed: " << ret << "\n";
        return 1;
      }

      std::map<ge::AscendString, ge::AscendString> global_options;
      global_options.emplace(ge::AscendString(ge::ir_option::SOC_VERSION), ge::AscendString(argv[6]));
      ret = ge::aclgrphBuildInitialize(global_options);
      if (ret != ge::GRAPH_SUCCESS) {
        std::cerr << "aclgrphBuildInitialize failed: " << ret << "\n";
        return 1;
      }

      std::map<ge::AscendString, ge::AscendString> build_options;
      build_options.emplace(ge::AscendString("input_format"), ge::AscendString("ND"));
      ge::ModelBufferData model;
      ret = ge::aclgrphBuildModel(graph, build_options, model);
      if (ret == ge::GRAPH_SUCCESS) {
        ret = ge::aclgrphSaveModel(argv[7], model);
      }
      ge::aclgrphBuildFinalize();
      if (ret != ge::GRAPH_SUCCESS) {
        std::cerr << "Build/Save failed: " << ret << "\n";
        return 1;
      }
      std::cout << "build_status=0 model_bytes=" << model.length << "\n";
      return 0;
    }
    '''
)
build_source_path = workdir / "tf_build_demo.cpp"
build_binary_path = workdir / "tf_build_demo"
build_source_path.write_text(build_source, encoding="utf-8")

build_compile_cmd = [
    "g++", "-std=c++17", "-O2",
    "-D_GLIBCXX_USE_CXX11_ABI=0", "-Dgoogle=ascend_private",
    "-I" + str(prefix / "include"), str(build_source_path),
]
for directory in library_dirs:
    build_compile_cmd.append("-L" + str(directory))
build_compile_cmd.extend([
    "-Wl,--no-as-needed", "-Wl,-rpath," + rpath,
    "-lfmk_parser", "-lge_compiler", "-lge_runner",
    "-lgraph", "-lgraph_base", "-lge_common_base",
    "-o", str(build_binary_path),
])
compile_result = subprocess.run(build_compile_cmd, text=True, capture_output=True)
if compile_result.returncode != 0:
    print(compile_result.stdout)
    print(compile_result.stderr)
    raise RuntimeError("阶段二 Build API 示例编译失败。")

om_stem = workdir / "tf_add"
om_path = Path(str(om_stem) + ".om")
if om_path.exists():
    om_path.unlink()
build_env = os.environ.copy()
build_env.setdefault("ASCEND_OPP_PATH", str(opp_path))
# 将 GE/TBE 初始化日志直接打印到 cell，便于定位 OPP 或 Python 依赖问题。
build_env.setdefault("ASCEND_SLOG_PRINT_TO_STDOUT", "1")
build_env["LD_LIBRARY_PATH"] = rpath + os.pathsep + build_env.get("LD_LIBRARY_PATH", "")
build_result = subprocess.run(
    [
        str(build_binary_path), str(model_path), input_shape, out_nodes,
        graph_name, input_data_names, soc_version, str(om_stem),
    ],
    text=True, capture_output=True, env=build_env,
)
if build_result.returncode != 0:
    print(build_result.stdout)
    print(build_result.stderr)
    print("提示：若日志包含 ModuleNotFoundError（例如 decorator），请补齐 CANN 主机侧 TBE/Python 依赖后重试。")
    raise RuntimeError("阶段二 Build API 运行失败。")
if not om_path.is_file() or om_path.stat().st_size == 0:
    raise RuntimeError("Build API 返回成功，但没有找到有效 OM 文件：{}".format(om_path))
print(build_result.stdout)
print("OM 文件：", om_path, "大小：", om_path.stat().st_size, "bytes")
print("[OK] 阶段二完成：Parser 生成的 Graph 已由 Build API 编译并保存为 OM")

### 4.3 阶段三：OM → ACL → NPU 执行（真实设备）

本阶段使用 CANN ACL 加载阶段二生成的 OM，按 `aclmdlDesc` 查询输入输出大小，并通过 `aclrtGetRunMode` 自动区分 `ACL_HOST` 与 `ACL_DEVICE`：Host 模式使用 H2D/D2H 拷贝，Device 模式直接访问 `aclrtMalloc` 地址，最后完成同步推理和数值对拍。

阶段三只适用于本单元内置的二输入 Add Graph：

~~~text
[1,2,3,4,5,6] + [10,20,30,40,50,60]
    → [11,22,33,44,55,66]
~~~

它需要真实的 NPU、驱动和运行时。cell 内会固定设置 `TF_RUN_NPU=1`，直接运行就会调用 `aclInit`、`aclrtSetDevice` 和 `aclmdlExecute`，无需提前手动设置环境变量。本示例输入和输出均为 `float32`；如果设置了 `TF_MODEL_PATH`，请不要直接使用本阶段的 Add 对拍代码，应按自己的模型 descriptor 修改输入和期望输出。

In [ ]:
# === 阶段 3：OM -> ACL -> NPU 执行与 NumPy 语义对拍 ===
import os

if not globals().get("is_builtin_tf_add", False):
    raise RuntimeError("阶段三的输入数据和期望输出只适用于阶段一内置的二输入 float32 Add 图。")
if "om_path" not in globals() or not Path(om_path).is_file():
    raise RuntimeError("请先成功执行阶段二，生成 OM 文件。")

# 本阶段就是 NPU 实战，cell 内固定启用；直接运行即可。
os.environ["TF_RUN_NPU"] = "1"
print("TF_RUN_NPU =", os.environ["TF_RUN_NPU"])

runner_source = textwrap.dedent(
    r'''
    #include <algorithm>
    #include <cmath>
    #include <cstdint>
    #include <cstring>
    #include <iostream>
    #include <string>
    #include <vector>
    #include "acl/acl.h"
    #include "acl/acl_mdl.h"
    #include "acl/acl_rt.h"

    namespace {
    bool CheckAcl(const char *operation, aclError status) {
      if (status == ACL_SUCCESS) return true;
      std::cerr << operation << " failed, ret=" << static_cast<int>(status);
      const char *message = aclGetRecentErrMsg();
      if (message != nullptr && message[0] != 0) {
        std::cerr << ", message=" << message;
      }
      std::cerr << "\n";
      return false;
    }

    void PrintTensorDesc(aclmdlDesc *desc, bool input, size_t index) {
      const size_t bytes = input ? aclmdlGetInputSizeByIndex(desc, index)
                                 : aclmdlGetOutputSizeByIndex(desc, index);
      const aclDataType dtype = input ? aclmdlGetInputDataType(desc, index)
                                      : aclmdlGetOutputDataType(desc, index);
      const aclFormat format = input ? aclmdlGetInputFormat(desc, index)
                                     : aclmdlGetOutputFormat(desc, index);
      aclmdlIODims dims{};
      const aclError dims_status = input ? aclmdlGetInputDims(desc, index, &dims)
                                         : aclmdlGetOutputDims(desc, index, &dims);
      std::cout << (input ? "input[" : "output[") << index
                << "] bytes=" << bytes
                << " dtype=" << static_cast<int>(dtype)
                << " format=" << static_cast<int>(format)
                << " dims_status=" << static_cast<int>(dims_status)
                << " dims=[";
      if (dims_status == ACL_SUCCESS) {
        for (size_t i = 0; i < dims.dimCount; ++i) {
          std::cout << (i == 0 ? "" : ",") << dims.dims[i];
        }
      }
      std::cout << "]\n";
    }

    struct Buffers {
      aclmdlDataset *dataset = nullptr;
      std::vector<void *> ptrs;
      std::vector<size_t> sizes;
    };

    void Destroy(Buffers *buffers) {
      if (buffers == nullptr || buffers->dataset == nullptr) return;
      const size_t n = aclmdlGetDatasetNumBuffers(buffers->dataset);
      for (size_t i = 0; i < n; ++i) {
        aclDataBuffer *buffer = aclmdlGetDatasetBuffer(buffers->dataset, i);
        if (buffer != nullptr) {
          void *ptr = aclGetDataBufferAddr(buffer);
          if (ptr != nullptr) (void)aclrtFree(ptr);
          (void)aclDestroyDataBuffer(buffer);
        }
      }
      (void)aclmdlDestroyDataset(buffers->dataset);
      buffers->dataset = nullptr;
    }

    bool Create(aclmdlDesc *desc, bool input, Buffers *buffers) {
      buffers->dataset = aclmdlCreateDataset();
      if (buffers->dataset == nullptr) return false;
      const size_t n = input ? aclmdlGetNumInputs(desc) : aclmdlGetNumOutputs(desc);
      for (size_t i = 0; i < n; ++i) {
        const size_t bytes = input ? aclmdlGetInputSizeByIndex(desc, i)
                                   : aclmdlGetOutputSizeByIndex(desc, i);
        void *ptr = nullptr;
        if (aclrtMalloc(&ptr, bytes, ACL_MEM_MALLOC_NORMAL_ONLY) != ACL_SUCCESS) return false;
        aclDataBuffer *buffer = aclCreateDataBuffer(ptr, bytes);
        if (buffer == nullptr || aclmdlAddDatasetBuffer(buffers->dataset, buffer) != ACL_SUCCESS) {
          if (buffer != nullptr) (void)aclDestroyDataBuffer(buffer);
          (void)aclrtFree(ptr);
          return false;
        }
        buffers->ptrs.push_back(ptr);
        buffers->sizes.push_back(bytes);
      }
      return true;
    }

    bool CopyInput(const std::vector<float> &source, size_t index,
                   const Buffers &inputs, aclrtRunMode run_mode) {
      const size_t bytes = source.size() * sizeof(float);
      if (index >= inputs.ptrs.size()) {
        std::cerr << "input[" << index << "] buffer is missing\n";
        return false;
      }
      if (inputs.sizes[index] < bytes) {
        std::cerr << "input[" << index << "] descriptor_bytes="
                  << inputs.sizes[index] << ", required_float32_bytes="
                  << bytes << "\n";
        return false;
      }
      if (!CheckAcl("aclrtMemset(input)",
                    aclrtMemset(inputs.ptrs[index], inputs.sizes[index], 0,
                                inputs.sizes[index]))) {
        return false;
      }
      if (run_mode == ACL_HOST) {
        return CheckAcl(
            "aclrtMemcpy(input H2D)",
            aclrtMemcpy(inputs.ptrs[index], inputs.sizes[index], source.data(),
                        bytes, ACL_MEMCPY_HOST_TO_DEVICE));
      }
      if (run_mode == ACL_DEVICE) {
        // Device 侧进程可直接访问 aclrtMalloc 返回的地址。
        std::memcpy(inputs.ptrs[index], source.data(), bytes);
        return true;
      }
      std::cerr << "unsupported aclrtRunMode=" << static_cast<int>(run_mode) << "\n";
      return false;
    }

    bool ReadOutput(const Buffers &outputs, aclrtRunMode run_mode,
                    std::vector<float> *actual) {
      const size_t bytes = outputs.sizes[0];
      if (run_mode == ACL_HOST) {
        return CheckAcl(
            "aclrtMemcpy(output D2H)",
            aclrtMemcpy(actual->data(), bytes, outputs.ptrs[0], bytes,
                        ACL_MEMCPY_DEVICE_TO_HOST));
      }
      if (run_mode == ACL_DEVICE) {
        // Device 侧进程直接读取 aclrtMalloc 输出地址。
        std::memcpy(actual->data(), outputs.ptrs[0], bytes);
        return true;
      }
      std::cerr << "unsupported aclrtRunMode=" << static_cast<int>(run_mode) << "\n";
      return false;
    }
    }  // namespace

    int main(int argc, char **argv) {
      if (argc != 3) {
        std::cerr << "usage: tf_acl_demo MODEL.om DEVICE_ID\n";
        return 2;
      }
      const int32_t device_id = static_cast<int32_t>(std::stoi(argv[2]));
      if (!CheckAcl("aclInit", aclInit(nullptr))) {
        return 1;
      }
      uint32_t count = 0;
      if (!CheckAcl("aclrtGetDeviceCount", aclrtGetDeviceCount(&count)) || count == 0) {
        std::cerr << "no available Ascend NPU device\n";
        (void)aclFinalize();
        return 3;
      }
      if (device_id < 0 || static_cast<uint32_t>(device_id) >= count) {
        std::cerr << "DEVICE_ID out of range: " << device_id
                  << ", available devices: " << count << "\n";
        (void)aclFinalize();
        return 2;
      }
      if (!CheckAcl("aclrtSetDevice", aclrtSetDevice(device_id))) {
        (void)aclFinalize();
        return 1;
      }
      aclrtRunMode run_mode = ACL_HOST;
      if (!CheckAcl("aclrtGetRunMode", aclrtGetRunMode(&run_mode))) {
        (void)aclrtResetDevice(device_id);
        (void)aclFinalize();
        return 1;
      }
      std::cout << "device_id=" << device_id << " run_mode="
                << (run_mode == ACL_HOST ? "ACL_HOST" : "ACL_DEVICE") << "\n";

      uint32_t model_id = 0;
      if (!CheckAcl("aclmdlLoadFromFile", aclmdlLoadFromFile(argv[1], &model_id))) {
        (void)aclrtResetDevice(device_id); (void)aclFinalize(); return 1;
      }
      aclmdlDesc *desc = aclmdlCreateDesc();
      if (desc == nullptr || aclmdlGetDesc(desc, model_id) != ACL_SUCCESS) {
        std::cerr << "aclmdlGetDesc failed\n";
        if (desc != nullptr) (void)aclmdlDestroyDesc(desc);
        (void)aclmdlUnload(model_id); (void)aclrtResetDevice(device_id); (void)aclFinalize();
        return 1;
      }
      if (aclmdlGetNumInputs(desc) != 2 || aclmdlGetNumOutputs(desc) != 1) {
        std::cerr << "the check expects two inputs and one output\n";
        (void)aclmdlDestroyDesc(desc); (void)aclmdlUnload(model_id);
        (void)aclrtResetDevice(device_id); (void)aclFinalize(); return 1;
      }
      for (size_t i = 0; i < aclmdlGetNumInputs(desc); ++i) {
        PrintTensorDesc(desc, true, i);
      }
      for (size_t i = 0; i < aclmdlGetNumOutputs(desc); ++i) {
        PrintTensorDesc(desc, false, i);
      }

      Buffers inputs, outputs;
      if (!Create(desc, true, &inputs) || !Create(desc, false, &outputs)) {
        Destroy(&inputs); Destroy(&outputs); (void)aclmdlDestroyDesc(desc);
        (void)aclmdlUnload(model_id); (void)aclrtResetDevice(device_id); (void)aclFinalize();
        return 1;
      }
      const std::vector<std::vector<float>> host_inputs = {
          {1.0F, 2.0F, 3.0F, 4.0F, 5.0F, 6.0F},
          {10.0F, 20.0F, 30.0F, 40.0F, 50.0F, 60.0F},
      };
      bool ok = true;
      for (size_t i = 0; i < host_inputs.size(); ++i) {
        if (!CopyInput(host_inputs[i], i, inputs, run_mode)) {
          ok = false;
          break;
        }
      }
      if (ok) {
        ok = CheckAcl("aclmdlExecute",
                      aclmdlExecute(model_id, inputs.dataset, outputs.dataset));
      }

      const std::vector<float> expected = {11.0F, 22.0F, 33.0F, 44.0F, 55.0F, 66.0F};
      float max_error = 0.0F;
      if (ok) {
        const size_t bytes = outputs.sizes[0];
        if (bytes < expected.size() * sizeof(float) || bytes % sizeof(float) != 0) {
          std::cerr << "output[0] descriptor_bytes=" << bytes
                    << ", required_float32_bytes="
                    << expected.size() * sizeof(float) << "\n";
          ok = false;
        } else {
          std::vector<float> actual(bytes / sizeof(float));
          if (!ReadOutput(outputs, run_mode, &actual)) {
            ok = false;
          } else {
            for (size_t i = 0; i < expected.size(); ++i) {
              if (!std::isfinite(actual[i])) {
                ok = false;
                break;
              }
              max_error = std::max(max_error, std::fabs(actual[i] - expected[i]));
            }
            std::cout << "output[0..5]=";
            for (size_t i = 0; i < expected.size(); ++i) {
              std::cout << (i == 0 ? "" : ",") << actual[i];
            }
            std::cout << "\nmax_abs_error=" << max_error << "\n";
            ok = ok && max_error <= 1.0e-4F;
          }
        }
      }
      Destroy(&inputs); Destroy(&outputs); (void)aclmdlDestroyDesc(desc);
      (void)aclmdlUnload(model_id); (void)aclrtResetDevice(device_id); (void)aclFinalize();
      if (!ok) {
        std::cerr << "ACL execution or numerical check failed\n";
        return 1;
      }
      std::cout << "[OK] TensorFlow .pb -> Parser -> OM -> ACL/NPU 数值对拍通过\n";
      return 0;
    }
    '''
)
runner_source_path = workdir / "tf_acl_demo.cpp"
runner_binary_path = workdir / "tf_acl_demo"
runner_source_path.write_text(runner_source, encoding="utf-8")

runner_compile_cmd = [
    "g++", "-std=c++17", "-O2",
    "-D_GLIBCXX_USE_CXX11_ABI=0", "-Dgoogle=ascend_private",
    "-I" + str(prefix / "include"), str(runner_source_path),
]
for directory in library_dirs:
    runner_compile_cmd.append("-L" + str(directory))
runner_compile_cmd.extend([
    "-Wl,--no-as-needed", "-Wl,-rpath," + rpath,
    "-lacl_mdl", "-lacl_rt", "-lge_runner", "-lge_compiler",
    "-lgraph", "-lgraph_base", "-lge_common_base",
    "-o", str(runner_binary_path),
])
compile_result = subprocess.run(runner_compile_cmd, text=True, capture_output=True)
if compile_result.returncode != 0:
    print(compile_result.stdout)
    print(compile_result.stderr)
    raise RuntimeError("阶段三 ACL 示例编译失败。")

try:
    runner_device_id = int(os.environ.get("TF_DEVICE_ID", "0"))
    if runner_device_id < 0:
        raise ValueError
except ValueError:
    raise RuntimeError("TF_DEVICE_ID 必须是非负整数。")
runner_env = os.environ.copy()
runner_env.setdefault("ASCEND_OPP_PATH", str(opp_path))
runner_env.setdefault("ASCEND_SLOG_PRINT_TO_STDOUT", "1")
runner_env["LD_LIBRARY_PATH"] = rpath + os.pathsep + runner_env.get("LD_LIBRARY_PATH", "")
runner_result = subprocess.run(
    [str(runner_binary_path), str(om_path), str(runner_device_id)],
    text=True, capture_output=True, env=runner_env,
)
print(runner_result.stdout)
print(runner_result.stderr)
if runner_result.returncode != 0:
    raise RuntimeError("阶段三 ACL/NPU 执行失败，详见上方日志。")
print("[OK] 阶段三完成：OM 已在 NPU 上加载、执行并完成数值对拍")

## 5. 接入约束与常见问题

| 问题类别 | 现象 / 关键日志 | 定位方向 |
| --- | --- | --- |
| **算子插件未注册** | `Check op[%s]'s type[%s] failed, it is not supported.` 或被转成 `frameworkop` | 算子插件 so 未加载成功，或缺少 TF→GE 的映射注册 |
| **算子原型未注册** | `IR for op[%s] optype[%s] is not registered.` / `have no ir factory` | 算子原型 so 未加载，或原型未编译进 so（用 `nm -D` 查符号表） |
| **输入 shape 不匹配** | 解析/编译报 shape 相关错误 | 检查 `INPUT_SHAPE` 与原始模型 Data 节点是否一致 |
| **输出节点选取错误** | 输出与预期不符 | 用 `OUT_NODES` 显式指定，并确认 name 为编译前模型中的真实节点名 |
| **图名不稳定** | 多次解析 Graph 名不同（含时间戳） | 用 `OUTPUT` 指定固定图名 |

定位通用入口（与 5.4 节一致）：

```bash
# 算子插件/原型加载成功的标志日志（关键字大小写和结尾以当前版本为准）：
#   Plugin load .../opp/built-in/framework/tensorflow/libops_all_plugin.so success.
#   OpsProtoManager plugin load .../opp/built-in/op_proto/libopsproto.so successfully.
# 第一条是 TensorFlow 框架算子适配插件；第二条是算子原型库。
# OPP 安装包可能包含多个插件/原型 so，实际文件名以当前安装目录为准。
# 加载失败：
#   dlopen failed, plugin name:%s. Message(%s).

nm -D xxx.so | grep <算子类型>     # 确认算子原型/插件是否注册进 so
```

> 解析（Parser）属于**编译前的前端转换**：报错多与「算子映射/原型注册」「输入输出节点/shape 配置」有关。GE 通用错误码与日志分析方法详见 **5.4 常见问题定位方法**。

## 6. 接入路径选型矩阵

回到选型：该用哪条路接入 GE。下面是 **TorchAir / TF Parser（含 TF Adapter）/ 手工构图** 的对比矩阵：

| 维度 | TorchAir（5.2） | TF Parser / TF Adapter（5.3） | GE 手工构图 |
| --- | --- | --- | --- |
| 适用框架 | PyTorch | TensorFlow | 任意（直接用 GE API） |
| 入口 | `torch.compile` + NPU backend | TF Adapter（在线）/ `aclgrphParseTensorFlow` + Build API（代码内）/ `atc`（CLI） | `op::` C++ 原型 / ES 极简 / Parser |
| 图来源 | Dynamo/FX 捕获 → AscendIR | TF GraphDef / `tf.function` → AscendIR；或 `.pb` 解析为内存 `Graph` | 用户逐算子构建 AscendIR |
| 编译时机 | 在线（运行时编译执行） | 在线（Adapter）/ 离线（Build API 或 ATC；两者为替代入口） | 在线（GeSession）/ 离线（Build API/ATC） |
| 代码改动量 | 极小（加 backend 即可） | 在线极小；Parser+Build 需写 C++，ATC CLI 可直接编译原始 `.pb` | 较大（手工搭图） |
| 灵活/可控性 | 中（受 Dynamo 捕获能力约束） | 中（受算子映射覆盖度约束） | 最高（完全自定义图结构） |
| 典型场景 | PyTorch 训练/推理图模式加速 | TF 模型迁移、固定图离线部署 | 自定义图、精细优化、实验 |

### 选型建议

<p align="left"><img src="./images/framework_selection.svg" alt="框架选择决策" width="65%"></p>

判断要点：

- 想最小改动接入 → 用对应框架的官方桥（TorchAir / TF Adapter）；
- 要独立部署、不依赖训练框架运行时 → 对原始 `.pb` 直接用 ATC，或用 Parser 得到 `Graph` 后接 Build API 生成 OM；
- 需要完全掌控图结构、做精细优化或实验 → 手工构图。

> 无论选哪条路，最终都收敛到同一张 **AscendIR 图**，进入统一的 GE 编译与执行流程。

## 课后练习

本节介绍了 TensorFlow 经 TF Adapter / Parser 接入 GE 的路径、`aclgrphParseTensorFlow` 用法，并通过 Parser → Build API → OM → ACL/NPU 三阶段实践完成离线闭环，请完成以下题目自测。

1. （判断题）`aclgrphParseTensorFlow` 用于将 TensorFlow 的 `.pb` 模型解析为 AscendIR `Graph`，返回值 `GRAPH_SUCCESS(0)` 表示成功。

2. （判断题）使用 `aclgrphParseTensorFlow` 解析得到的 Graph 名是固定不变的，多次调用不会变化。

3. （单选题）若使用 ATC CLI 将 TensorFlow `.pb` 模型离线编译为 OM，正确的路径是？
    A. TF 脚本 → TorchAir → GE
    B. model.pb → ATC（`--framework=3`）→ OM → ACL
    C. model.pb → torch.compile → OM
    D. TF 脚本 → ONNX → aclgrphParseTensorFlow

4. （单选题）以下关于 `aclgrphParseTensorFlow` 的 `parser_params` 描述，哪个正确？
    A. key/value 必须是 int 类型
    B. key 为参数类型、value 为参数值，均为 `AscendString` 格式
    C. 只能传入一个参数
    D. 该参数是输出参数

5. （多选题）以下哪些是 Parser 支持的配置参数？
    A. `INPUT_SHAPE`（指定输入 shape，支持静态/范围/标量）
    B. `OUT_NODES`（指定输出节点）
    C. `OUTPUT`（指定转图后计算图名称）
    D. `ENABLE_SCOPE_FUSION_PASSES`（指定生效的 scope 融合规则，仅 TF Parser 支持）

6. （单选题）模型转换时某算子被转成了 `frameworkop` 类型 / 报 `it is not supported`，最可能的原因是？
    A. 输入 shape 配置错误
    B. 算子插件 so 未加载成功或缺少映射注册
    C. 输出节点未指定
    D. Graph 名包含时间戳

7. （多选题）关于 TorchAir / TF Parser / 手工构图的选型，以下哪些正确？
    A. PyTorch 模型优先用 TorchAir（torch.compile + NPU backend）
    B. 已导出的 TF `.pb` 模型可直接用 ATC（`--framework=3`）离线编译；若先调用 Parser API，则应改接 GE Build API
    C. 需要完全掌控图结构、做精细优化时，可用 GE 手工构图
    D. 三种方式最终都收敛为同一种 AscendIR 图，进入统一的 GE 编译执行流程

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/05.03_answer.txt